<a href="https://colab.research.google.com/github/RohitKhobare/ML-Lab-Assignments/blob/main/ML_LA3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import pandas as pd
import numpy as np
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping


np.random.seed(42)
tf.random.set_seed(42)


print("Downloading Bank Customer Churn dataset from Kaggle...")
path = kagglehub.dataset_download("barelydedicated/bank-customer-churn-modeling")

csv_file = [os.path.join(path, f) for f in os.listdir(path) if f.endswith('.csv')][0]
df = pd.read_csv(csv_file)
print(f"Dataset loaded successfully! Initial shape: {df.shape}")

df = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])

X = df.drop(columns=['Exited'])
y = df['Exited']

categorical_features = ['Geography', 'Gender']
numerical_features = ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ]
)

X_train_scaled = preprocessor.fit_transform(X_train)
X_test_scaled = preprocessor.transform(X_test)


model = Sequential([
    Dense(units=32, activation='relu', kernel_initializer='he_normal', input_dim=X_train_scaled.shape[1]),
    Dropout(0.2),
    Dense(units=16, activation='relu', kernel_initializer='he_normal'),
    Dropout(0.2),
    Dense(units=8, activation='relu', kernel_initializer='he_normal'),
    Dense(units=1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("\nTraining Neural Network Model...")
history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    batch_size=32,
    epochs=100,
    callbacks=[early_stop],
    verbose=0
)

y_pred_probs = model.predict(X_test_scaled)
y_pred = (y_pred_probs > 0.5).astype(int)

acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n" + "="*50)
print("              EVALUATION RESULTS              ")
print("="*50)
print(f"Accuracy Score: {acc * 100:.2f}%\n")
print("Confusion Matrix:")
print(cm)
print("\nConfusion Matrix Breakdown:")
print(f"True Negatives (Retained correctly)  : {cm[0][0]}")
print(f"False Positives (Incorrectly flagged) : {cm[0][1]}")
print(f"False Negatives (Missed churners)    : {cm[1][0]}")
print(f"True Positives (Churned correctly)   : {cm[1][1]}")
print("="*50)

print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained (0)', 'Churned (1)']))

Using Colab cache for faster access to the 'bank-customer-churn-modeling' dataset.
Dataset loaded successfully! Initial shape: (10000, 14)

Training Neural Network Model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step

              EVALUATION RESULTS              
Accuracy Score: 85.85%

Confusion Matrix:
[[1524   69]
 [ 214  193]]

Confusion Matrix Breakdown:
True Negatives (Retained correctly)  : 1524
False Positives (Incorrectly flagged) : 69
False Negatives (Missed churners)    : 214
True Positives (Churned correctly)   : 193

Detailed Classification Report:
              precision    recall  f1-score   support

Retained (0)       0.88      0.96      0.92      1593
 Churned (1)       0.74      0.47      0.58       407

    accuracy                           0.86      2000
   macro avg       0.81      0.72      0.75      2000
weighted avg       0.85      0.86      0.85      2000

